# Building a Chatbot with Memory 💬

Here's the single most important thing to understand before building *any* AI chat app:

> **An LLM has no memory.** Every time you call it, it's a completely blank slate. It does *not* remember your previous message.

Sounds broken, right? How does ChatGPT remember your conversation then? The trick is beautifully simple — and it's just a Python **list**. By the end of this notebook you'll know exactly how every chatbot on earth "remembers" things.

This lesson leans entirely on stuff you already know: **lists, dicts, functions** (lesson 1 & 2), the **LLM call** (lesson 4), and **try/except** (lesson 5). Almost nothing new — just a clever idea.

## 1. Prove it: the LLM forgets instantly

Let's catch the model being forgetful. First the usual setup:

In [2]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()

Tell it our name in one call...

In [3]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Hi! My name is Dhaval. Please remember it."
)
print(response.text)

Hi Dhaval! Got it. I'll remember your name.


...now ask it our name in a **separate** call:

In [4]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What is my name?"
)
print(response.text)

As an AI, I don't have access to any personal information about you, so I don't know your name. You'll have to tell me!


😬 Busted. It has no clue. The LLM has the memory of a goldfish 🐠 — actually worse, because a goldfish at least remembers the last 3 seconds.

**Why?** Each `generate_content` call is independent. The model never saw your first message when answering the second. It's like texting someone who deletes the chat after every single text.

## 2. The fix: keep the whole conversation in a list

The secret: we **remember the conversation ourselves** in a list, and re-send the *entire history* on every call. The model can't remember — so we remind it of everything, every time.

Each message is a little **dict** with two keys: who said it (`role`) and what they said (`parts`). Gemini uses `"user"` for us and `"model"` for the AI.

> 📝 The `"parts": [{"text": ...}]` wrapper looks fussy, but it's just Gemini's required format for a message. Write it once in a function and forget about it.

In [5]:
conversation = []   # this list IS the chatbot's memory

def chat(user_message):
    # 1. add what the user said to memory
    conversation.append({"role": "user", "parts": [{"text": user_message}]})

    # 2. send the ENTIRE conversation so far
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=conversation
        )
        reply = response.text
    except Exception as e:
        return f"Sorry, something went wrong: {e}"

    # 3. add the AI's reply to memory too, so it remembers next time
    conversation.append({"role": "model", "parts": [{"text": reply}]})
    return reply

Now let's run the exact same test — name first, then ask:

In [6]:
print(chat("Hi! My name is Dhaval. Please remember it."))
print("---")
print(chat("What is my name?"))

Hi Dhaval! Got it, I'll remember that.

Nice to meet you! How can I help you today?
---
Your name is Dhaval.


🎉 It remembers! Nothing magical happened — we just handed the model the back-story each time.

Let's peek inside its "brain" (our list) to see what's actually stored:

In [7]:
for message in conversation:
    print(message["role"], "->", message["parts"][0]["text"])
    print()

user -> Hi! My name is Dhaval. Please remember it.

model -> Hi Dhaval! Got it, I'll remember that.

Nice to meet you! How can I help you today?

user -> What is my name?

model -> Your name is Dhaval.



That growing list is *the entire concept of chatbot memory*. Every chat product you've used is doing this under the hood.

> 💡 **Good to know:** the `google-genai` library also has a shortcut — `client.chats.create(...)` — that manages this list *for* you. We did it by hand on purpose, so you actually understand what that shortcut is hiding. (Remember the classes lesson? Same idea — now you know what's under the hood.)

## 3. Make it talk back: a real chat app

A real chatbot needs to keep taking input in a loop. A `while True` + `input()` loop is clunky inside a notebook, so the proper interactive version lives in **`app.py`** (a Streamlit app) right next to this notebook.

Run it from your terminal:

```bash
streamlit run app.py
```

The app uses the **exact same idea** you just learned — a conversation list that grows with every turn. The only new piece is `st.session_state`, which keeps that list alive between button clicks (otherwise Streamlit would wipe it on every interaction).

Open `app.py`, read it, and tweak it. It's barely 25 lines. 🚀

## 🏋️ Exercise: A movie-recommendation buddy that remembers your taste

Build a chatbot that uses memory to give *personalized* answers.

1. Start a fresh `conversation = []` and reuse the `chat()` function from above.
2. Have this conversation, one `chat()` call at a time:
   - `"I love sci-fi and thrillers, but I can't stand horror movies."`
   - `"Suggest me a movie to watch tonight."`
3. Check that its suggestion respects what you said earlier (sci-fi/thriller, **not** horror). That's memory doing its job!
4. Then ask: `"Why did you pick that one?"` — it should explain using your stated taste.

**Bonus 1:** add a `reset()` function that empties the conversation list — a "new chat" button.

**Bonus 2:** give your bot a personality. Pass a `config` to `generate_content`:
```python
from google.genai import types
config = types.GenerateContentConfig(
    system_instruction="You are a witty movie buff who loves making puns."
)
# ...add config=config to the generate_content call
```

Have fun — try to make it forget on purpose by clearing the list, then watch it ask "who are you?" all over again. 😄

In [ ]:
# Your solution here
